# CredResolve Data Analyst Assignment

**Dataset:** Synthetic Collections Analytics Dataset (Seed=42, ~639k rows, 18 tables)
**Submission:** 31 October 2026

## Executive Summary
CredResolve claimed an 11% month-on-month improvement in debt recovery.
This notebook investigates that claim rigorously.

LIMITATION: Event data covers Jan-Aug 2026 (approx 7.5 months), not 12 months.
Aug 2026 is a partial month (8 days only). All trends are disclosed accordingly.


In [ ]:
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.labelcolor': '#e0e0e0', 'xtick.color': '#b0b0b0',
    'ytick.color': '#b0b0b0', 'text.color': '#e0e0e0',
    'grid.color': '#2a2d3a', 'grid.linestyle': '--', 'grid.alpha': 0.5,
    'figure.dpi': 120
})
PALETTE = ['#4fc3f7','#81c784','#ffb74d','#f06292','#ce93d8','#80cbc4','#fff176']
sns.set_palette(PALETTE)
DATA = Path('data') if Path('data').exists() else Path('../data')
print('Data path:', DATA.resolve())
print('CSV files:', len(list(DATA.glob('*.csv'))))


---
## Section 1: Load and Profile All Tables

In [ ]:
TABLES = [
    'borrowers','accounts','agents','agent_sessions','campaigns',
    'daily_targeting','calls','call_attempts','call_dispositions',
    'whatsapp_events','sms_events','field_visits','promises_to_pay',
    'payments','vendor_telephony','complaints','account_status_history'
]
raw = {}
for t in TABLES:
    raw[t] = pd.read_csv(DATA / f'{t}.csv')
    print(f'  {t:35s}  {len(raw[t]):>7,} rows  {raw[t].shape[1]:>2} cols')


In [ ]:
rows = []
for name, df in raw.items():
    rows.append({
        'table': name, 'rows': len(df), 'cols': len(df.columns),
        'exact_dup_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
        'missing_pct': round(df.isna().sum().sum() / max(df.size, 1) * 100, 2),
    })
profile_df = pd.DataFrame(rows).sort_values('rows', ascending=False)
print(profile_df.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Data Quality Overview', fontsize=14, fontweight='bold', color='#e0e0e0')
dup = profile_df[profile_df['exact_dup_rows'] > 0].sort_values('exact_dup_rows')
axes[0].barh(dup['table'], dup['exact_dup_rows'], color='#f06292')
axes[0].set_title('Tables with Exact Duplicate Rows', color='#e0e0e0'); axes[0].grid(axis='x')
miss = profile_df[profile_df['missing_cells'] > 0].sort_values('missing_cells')
axes[1].barh(miss['table'], miss['missing_pct'], color='#ffb74d')
axes[1].set_title('Tables with Missing Data Percent', color='#e0e0e0'); axes[1].grid(axis='x')
plt.tight_layout()
plt.savefig('data_quality_overview.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


In [ ]:
event_tables = ['calls','call_attempts','payments','promises_to_pay',
                'whatsapp_events','sms_events','field_visits','complaints']
coverage = []
for name in event_tables:
    ts = pd.to_datetime(raw[name]['event_at'], errors='coerce')
    coverage.append({
        'table': name, 'valid_timestamps': int(ts.notna().sum()),
        'min_event': str(ts.min().date()) if ts.notna().any() else None,
        'max_event': str(ts.max().date()) if ts.notna().any() else None,
        'days_covered': (ts.max() - ts.min()).days if ts.notna().any() else 0
    })
print(pd.DataFrame(coverage).to_string(index=False))
print()
print('LIMITATION: Data covers Jan-Aug 2026 (~7.5 months), not 12.')
print('Aug 2026 partial (8 days). Month-on-month trends window is Jan-Jul 2026 only.')


---
## Section 2: Payment Forensics and Golden Layer

In [ ]:
pay = raw['payments'].copy()
pay['amount'] = pd.to_numeric(pay['amount'], errors='coerce')
pay['event_at'] = pd.to_datetime(pay['event_at'], errors='coerce')
pay['payment_status'] = pay['payment_status'].str.upper().str.strip()
print('RAW PAYMENT TABLE')
print(f'Total rows: {len(pay):,}')
print(f'Exact dup rows: {int(pay.duplicated().sum()):,}')
print()
print(pay['payment_status'].value_counts())
print()
print(pay['amount'].describe().round(2))


In [ ]:
# payment_reference alone is NOT a safe dedup key
ref_audit = pay.groupby('payment_reference', dropna=False).agg(
    row_count=('payment_id', 'size'),
    distinct_accts=('account_id', 'nunique'),
    distinct_amts=('amount', 'nunique'),
    distinct_statuses=('payment_status', 'nunique'),
    amount_sum=('amount', 'sum')
).reset_index()
repeated = ref_audit[ref_audit['row_count'] > 1]
print(f'Unique payment references: {len(ref_audit):,}')
print(f'References appearing more than once: {len(repeated):,}')
print(f'Rows in repeated-reference groups: {int(repeated["row_count"].sum()):,}')
print(f'Refs spanning multiple accounts: {int((repeated["distinct_accts"] > 1).sum()):,}')
print(f'Refs with multiple statuses: {int((repeated["distinct_statuses"] > 1).sum()):,}')
print()
print('Top 10 repeated references:')
print(repeated.sort_values('row_count', ascending=False).head(10).to_string(index=False))


In [ ]:
# GOLDEN PAYMENT LAYER
# Step 1: remove byte-identical duplicates
pay_deduped = pay.drop_duplicates().copy()
print(f'After exact-dup removal: {len(pay_deduped):,} rows (removed {len(pay)-len(pay_deduped):,})')

# Step 2: flag groups with any REVERSED row
reversal_refs = set(
    pay_deduped.loc[pay_deduped['payment_status'] == 'REVERSED', 'payment_reference'].dropna()
)
pay_deduped['has_reversal'] = pay_deduped['payment_reference'].isin(reversal_refs)

# Step 3: golden SUCCESS excludes reversal groups; dedup on payment_id
golden_success = pay_deduped[
    (pay_deduped['payment_status'] == 'SUCCESS') &
    (~pay_deduped['has_reversal'])
].copy()
golden_success = golden_success.sort_values('event_at').drop_duplicates(subset='payment_id', keep='last')
recovered_accts = set(golden_success['account_id'].dropna())

print(f'Golden successful payments: {len(golden_success):,}')
print(f'Net recovery amount:        Rs {golden_success["amount"].sum():,.0f}')
print(f'Reversal-affected refs excluded: {len(reversal_refs):,}')


In [ ]:
golden_success['month'] = golden_success['event_at'].dt.to_period('M')
monthly = golden_success.groupby('month').agg(
    payments=('payment_id', 'count'), net_recovered=('amount', 'sum')
).reset_index()
monthly['month_str'] = monthly['month'].astype(str)
monthly_full = monthly[monthly['month_str'] < '2026-08'].copy()
monthly_full['mom_pct'] = monthly_full['net_recovered'].pct_change() * 100
print('Monthly Net Recovery (full months only):')
print(monthly_full[['month_str','payments','net_recovered','mom_pct']].to_string(index=False))
print()
avg_mom = monthly_full['mom_pct'].dropna().mean()
print(f'Average MoM change Jan-Jul 2026: {avg_mom:+.1f}%')
if avg_mom >= 11:
    print('Data SUPPORTS the 11% MoM claim on average.')
elif avg_mom > 0:
    print(f'Positive trend ({avg_mom:.1f}%) but does NOT reach 11% consistently.')
else:
    print('Data does NOT support the 11% MoM improvement claim.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Monthly Recovery Analysis', fontsize=14, fontweight='bold', color='#e0e0e0')
axes[0].bar(monthly['month_str'], monthly['net_recovered'] / 1e7, color='#4fc3f7', alpha=0.85)
axes[0].set_title('Net Recovery per Month (Rs Crore)', color='#e0e0e0')
axes[0].tick_params(axis='x', rotation=40); axes[0].grid(axis='y')
mom_vals = monthly_full['mom_pct'].dropna().values
months_x = list(monthly_full['month_str'].iloc[1:])
bar_colors = ['#81c784' if v >= 0 else '#f06292' for v in mom_vals]
axes[1].bar(months_x, mom_vals, color=bar_colors, alpha=0.85)
axes[1].axhline(0, color='#aaa', linewidth=0.8)
axes[1].axhline(11, color='#ffb74d', linewidth=1.5, linestyle='--', label='Claimed 11%')
axes[1].set_title('Month-on-Month Recovery Change Percent', color='#e0e0e0')
axes[1].tick_params(axis='x', rotation=40); axes[1].legend(); axes[1].grid(axis='y')
plt.tight_layout()
plt.savefig('monthly_recovery.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


---
## Section 3: Recovery Rate with Fixed Denominator

In [ ]:
acc = raw['accounts'].copy()
acc['outstanding_amount'] = pd.to_numeric(acc['outstanding_amount'], errors='coerce')
acc['dpd'] = pd.to_numeric(acc['dpd'], errors='coerce')
print(acc['status'].value_counts()); print(); print(acc['risk_segment'].value_counts())
eligible = acc[acc['status'] != 'PAID'].copy()
eligible['recovered'] = eligible['account_id'].isin(recovered_accts)
print(f'Eligible accounts: {len(eligible):,}')
print(f'Overall recovery rate: {eligible["recovered"].mean()*100:.1f}%')


In [ ]:
seg_rate = eligible.groupby('risk_segment').agg(
    total=('account_id', 'count'), recovered=('recovered', 'sum')
).reset_index()
seg_rate['recovery_rate_pct'] = (seg_rate['recovered'] / seg_rate['total'] * 100).round(1)
print(seg_rate.to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Recovery Rate Analysis', fontsize=14, fontweight='bold', color='#e0e0e0')
sns.barplot(data=seg_rate, x='risk_segment', y='recovery_rate_pct', palette=PALETTE, ax=axes[0])
axes[0].set_title('Recovery Rate by Risk Segment', color='#e0e0e0'); axes[0].grid(axis='y')
lr = eligible.groupby('loan_type').agg(t=('account_id','count'), r=('recovered','sum')).reset_index()
lr['rate'] = lr['r'] / lr['t'] * 100
sns.barplot(data=lr, x='loan_type', y='rate', palette=PALETTE, ax=axes[1])
axes[1].set_title('Recovery Rate by Loan Type', color='#e0e0e0')
axes[1].tick_params(axis='x', rotation=15); axes[1].grid(axis='y')
plt.tight_layout()
plt.savefig('recovery_by_segment.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


---
## Section 4: Channel Effectiveness

In [ ]:
calls = raw['calls'].copy()
calls['event_at'] = pd.to_datetime(calls['event_at'], errors='coerce')
wa  = raw['whatsapp_events'].copy()
sms = raw['sms_events'].copy()
fv  = raw['field_visits'].copy()

print('CALLS:', calls['call_status'].value_counts().to_dict())
print('WA events:', wa['event_type'].value_counts().to_dict())
print('SMS events:', sms['event_type'].value_counts().to_dict())
print('Field visit outcomes:', fv['outcome'].value_counts().to_dict())

answered    = set(calls.loc[calls['call_status']=='ANSWERED', 'account_id'].dropna())
wa_clicked  = set(wa.loc[wa['event_type']=='PAYMENT_CLICK', 'account_id'].dropna())
sms_clicked = set(sms.loc[sms['event_type']=='CLICKED', 'account_id'].dropna())
fv_all      = set(fv['account_id'].dropna())
fv_ptp      = set(fv.loc[fv['outcome'].isin(['PAID','PTP']), 'account_id'].dropna())
print(f'Call answer rate: {len(answered)/len(calls)*100:.1f}%')


In [ ]:
ch_df = pd.DataFrame({
    'Channel': ['Phone Answered', 'WhatsApp Clicked', 'SMS Clicked', 'Field PTP or Paid'],
    'Engaged': [len(answered), len(wa_clicked), len(sms_clicked), len(fv_all)],
    'Paid':    [len(answered & recovered_accts), len(wa_clicked & recovered_accts),
                len(sms_clicked & recovered_accts), len(fv_ptp & recovered_accts)]
})
ch_df['Conversion_pct'] = (ch_df['Paid'] / ch_df['Engaged'] * 100).round(1)
print(ch_df.to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Channel Effectiveness', fontsize=14, fontweight='bold', color='#e0e0e0')
sns.barplot(data=ch_df, x='Channel', y='Conversion_pct', palette=PALETTE[:4], ax=axes[0])
axes[0].set_title('Conversion Rate by Channel Percent', color='#e0e0e0')
axes[0].tick_params(axis='x', rotation=15); axes[0].grid(axis='y')
axes[1].bar(ch_df['Channel'], ch_df['Engaged'], color=PALETTE[0], label='Engaged', alpha=0.8)
axes[1].bar(ch_df['Channel'], ch_df['Paid'], color=PALETTE[1], label='Paid', alpha=0.9)
axes[1].set_title('Engaged vs Paid Accounts by Channel', color='#e0e0e0')
axes[1].tick_params(axis='x', rotation=15); axes[1].legend(); axes[1].grid(axis='y')
plt.tight_layout()
plt.savefig('channel_effectiveness.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


---
## Section 5: Agent and Team Performance

In [ ]:
agents = raw['agents'].copy()
ptp    = raw['promises_to_pay'].copy()
ptp['promised_amount'] = pd.to_numeric(ptp['promised_amount'], errors='coerce')
print('Agent teams:'); print(agents['team'].value_counts())
print(); print('Agent status:'); print(agents['status'].value_counts())
agent_calls = calls.groupby('agent_id').agg(
    total=('call_id', 'count'),
    answered=('call_status', lambda x: (x == 'ANSWERED').sum())
).reset_index()
agent_calls['answer_rate'] = agent_calls['answered'] / agent_calls['total'] * 100
ptp_ag = ptp.groupby('agent_id').agg(
    ptps=('ptp_id', 'count'),
    kept=('status', lambda x: (x == 'KEPT').sum())
).reset_index()
ptp_ag['kept_rate'] = ptp_ag['kept'] / ptp_ag['ptps'] * 100
agent_full = (agent_calls
    .merge(ptp_ag, on='agent_id', how='left')
    .merge(agents[['agent_id','team','status']], on='agent_id', how='left'))
team_full = agent_full.groupby('team').agg(
    agents=('agent_id', 'count'), total_calls=('total', 'sum'),
    avg_answer_rate=('answer_rate', 'mean'),
    total_ptps=('ptps', 'sum'), total_kept=('kept', 'sum')
).reset_index().dropna(subset=['team'])
team_full['ptp_kept_rate'] = team_full['total_kept'] / team_full['total_ptps'] * 100
print(team_full.round(1).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Team Performance', fontsize=14, fontweight='bold', color='#e0e0e0')
sns.barplot(data=team_full, x='team', y='avg_answer_rate', palette=PALETTE, ax=axes[0])
axes[0].set_title('Avg Call Answer Rate by Team', color='#e0e0e0'); axes[0].grid(axis='y')
sns.barplot(data=team_full, x='team', y='ptp_kept_rate', palette=PALETTE[::-1], ax=axes[1])
axes[1].set_title('PTP Kept Rate by Team', color='#e0e0e0'); axes[1].grid(axis='y')
plt.tight_layout()
plt.savefig('team_performance.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


---
## Section 6: Promise-to-Pay Fulfillment

In [ ]:
print(f'Total PTPs: {len(ptp):,}')
print(ptp['status'].value_counts())
ptp_rate = ptp['status'].value_counts(normalize=True) * 100
print(f'Kept: {ptp_rate.get("KEPT",0):.1f}%   Broken: {ptp_rate.get("BROKEN",0):.1f}%')
print(); print('By source:'); print(ptp['source'].value_counts())
src_rate = ptp.groupby('source').apply(
    lambda x: (x['status'] == 'KEPT').sum() / len(x) * 100
).reset_index()
src_rate.columns = ['source', 'kept_rate']
print(src_rate.sort_values('kept_rate', ascending=False).to_string(index=False))
kept_accts = set(ptp.loc[ptp['status'] == 'KEPT', 'account_id'].dropna())
print(f'Kept PTP accounts that actually paid: {len(kept_accts & recovered_accts):,} / {len(kept_accts):,}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('PTP Analysis', fontsize=14, fontweight='bold', color='#e0e0e0')
pc = ptp['status'].value_counts()
axes[0].pie(pc.values, labels=pc.index, colors=PALETTE[:len(pc)],
            autopct='%1.1f%%', textprops={'color':'#e0e0e0'})
axes[0].set_title('PTP Status Distribution', color='#e0e0e0')
sns.barplot(data=src_rate.sort_values('kept_rate', ascending=False),
            x='source', y='kept_rate', palette=PALETTE, ax=axes[1])
axes[1].set_title('PTP Kept Rate by Source Percent', color='#e0e0e0')
axes[1].tick_params(axis='x', rotation=15); axes[1].grid(axis='y')
plt.tight_layout()
plt.savefig('ptp_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


---
## Section 7: Complaint Risk

In [ ]:
comp = raw['complaints'].copy()
comp['event_at'] = pd.to_datetime(comp['event_at'], errors='coerce')
comp['resolution_at'] = pd.to_datetime(comp['resolution_at'], errors='coerce')
comp['resolution_days'] = (comp['resolution_at'] - comp['event_at']).dt.days
print(f'Total complaints: {len(comp):,}')
print(comp['complaint_type'].value_counts())
print(comp['severity'].value_counts())
print(f'Critical: {(comp["severity"]=="CRITICAL").sum():,}')
print(f'Harassment: {(comp["complaint_type"]=="HARASSMENT").sum():,}')
print(f'DND violations: {(comp["complaint_type"]=="DND").sum():,}')
print(comp.groupby('severity')['resolution_days'].mean().round(1))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Complaint Risk Analysis', fontsize=14, fontweight='bold', color='#e0e0e0')
tc = comp['complaint_type'].value_counts()
axes[0].barh(tc.index, tc.values, color=PALETTE[:len(tc)])
axes[0].set_title('Complaint Count by Type', color='#e0e0e0'); axes[0].grid(axis='x')
comp['month'] = comp['event_at'].dt.to_period('M')
cmt = comp.groupby(['month','severity']).size().unstack(fill_value=0).reset_index()
cmt['month_str'] = cmt['month'].astype(str)
for i, sev in enumerate([c for c in ['CRITICAL','HIGH','MEDIUM','LOW'] if c in cmt.columns]):
    axes[1].plot(cmt['month_str'], cmt[sev], label=sev, color=PALETTE[i], marker='o', linewidth=2)
axes[1].set_title('Monthly Complaints by Severity', color='#e0e0e0')
axes[1].tick_params(axis='x', rotation=40); axes[1].legend(); axes[1].grid()
plt.tight_layout()
plt.savefig('complaint_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


---
## Section 8: Statistical Investigation of the 11% MoM Claim

In [ ]:
print('STATISTICAL TESTS')
monthly_vals = monthly_full['net_recovered'].values
x = np.arange(len(monthly_vals))
slope, intercept, r_val, p_val, std_err = stats.linregress(x, monthly_vals)
print(f'Linear trend: slope=Rs {slope:,.0f}/month  R2={r_val**2:.3f}  p={p_val:.4f}')
if p_val < 0.05:
    direction = 'UPWARD' if slope > 0 else 'DOWNWARD'
    print(f'  Statistically significant {direction} trend (p<0.05)')
else:
    print('  No statistically significant trend (p>=0.05)')
print()
mom_arr = monthly_full['mom_pct'].dropna().values
t1, p1 = stats.ttest_1samp(mom_arr, 0)
print(f't-test H0 MoM=0: mean={mom_arr.mean():.2f}%  t={t1:.3f}  p={p1:.4f}')
t2, p2 = stats.ttest_1samp(mom_arr, 11)
print(f't-test H0 MoM=11%: t={t2:.3f}  p={p2:.4f}')
if p2 < 0.05:
    print('Data significantly DIFFERS from the 11% claim.')
else:
    print('Cannot reject that data is consistent with 11% MoM growth.')


In [ ]:
print('MIX EFFECT ANALYSIS')
print('Check if growth is operational vs mix-driven (easy accounts growing)')
golden_success['month_str'] = golden_success['event_at'].dt.to_period('M').astype(str)
payers_seg = golden_success.merge(acc[['account_id','risk_segment']], on='account_id', how='left')
seg_mix = payers_seg.groupby(['month_str','risk_segment']).size().unstack(fill_value=0)
seg_mix_pct = seg_mix.div(seg_mix.sum(axis=1), axis=0) * 100
print(seg_mix_pct.round(1).to_string())
print()
print('Stable or growing HIGH/NPA share => operational improvement is real.')
print('Growing LOW/MEDIUM share => may be mix effect, not genuine improvement.')


---
## Section 9: Investment Recommendation

In [ ]:
total_outstanding = acc['outstanding_amount'].sum()
total_recovered   = golden_success['amount'].sum()
overall_pct       = len(golden_success['account_id'].unique()) / len(eligible) * 100

print('INVESTMENT RECOMMENDATION SUMMARY')
print(f'Portfolio outstanding:     Rs {total_outstanding:>15,.0f}')
print(f'Net recovery (7.5 months): Rs {total_recovered:>15,.0f}')
print(f'Portfolio recovery rate:   {overall_pct:.1f}%')
print()
print('KEY FINDINGS:')
print('  1. Calls: ~91k events but ~20% answer rate => ROI improvement possible')
print('  2. WhatsApp PAYMENT_CLICK = highest-intent digital signal')
print('  3. PTP broken rate ~25% = predictable cash-flow leakage')
print('  4. 1,126 harassment + 1,152 DND complaints = regulatory risk')
print('  5. 11% MoM claim is data-period and denominator sensitive')
print()
print('RECOMMENDED INVESTMENTS:')
print('  1. WhatsApp-first outreach automation (low cost, high-intent signal)')
print('  2. PTP follow-up automation (reduce broken PTP rate from ~25%)')
print('  3. Agent compliance training (harassment and DND reduction)')
print('  4. Risk-stratified channel routing:')
print('       HIGH/NPA => Field visits, MEDIUM => SMS, ALL => WhatsApp')
print('  5. Data engineering: dedup pipeline + timezone normalization')
print()
print('COUNTERFACTUAL A/B TEST DESIGN:')
print('  Treatment: WhatsApp-first + PTP automation cohort')
print('  Control:   Current call-first cohort (status quo)')
print('  Stratify by: risk_segment, DPD bucket, loan_type, outstanding quintile')
print('  Primary KPI: 60-day payment rate')
print('  Secondary:   Cost per recovered account, PTP kept rate, complaints rate')
print('  Duration:    Min 3 months for statistical power')


---
## Section 10: Final KPI Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('CredResolve - Collections Analytics Dashboard',
             fontsize=16, fontweight='bold', color='#4fc3f7', y=0.98)
gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0:2]); ax1.set_facecolor('#1a1d27')
ax1.bar(monthly['month_str'], monthly['net_recovered'] / 1e7, color='#4fc3f7', alpha=0.85)
ax1.set_title('Net Monthly Recovery (Rs Cr)', color='#e0e0e0')
ax1.tick_params(axis='x', rotation=30, colors='#b0b0b0'); ax1.grid(axis='y', alpha=0.3)

ax2 = fig.add_subplot(gs[0, 2]); ax2.set_facecolor('#1a1d27')
ax2.bar(seg_rate['risk_segment'], seg_rate['recovery_rate_pct'], color=PALETTE[:4])
ax2.set_title('Recovery Rate by Segment', color='#e0e0e0')
ax2.tick_params(colors='#b0b0b0'); ax2.grid(axis='y', alpha=0.3)

ax3 = fig.add_subplot(gs[1, 0]); ax3.set_facecolor('#1a1d27')
ax3.bar(ch_df['Channel'], ch_df['Conversion_pct'], color=PALETTE[:4])
ax3.set_title('Channel Conversion Rate Percent', color='#e0e0e0')
ax3.tick_params(axis='x', rotation=20, colors='#b0b0b0', labelsize=7); ax3.grid(axis='y', alpha=0.3)

ax4 = fig.add_subplot(gs[1, 1]); ax4.set_facecolor('#1a1d27')
pc2 = ptp['status'].value_counts()
ax4.pie(pc2.values, labels=pc2.index, colors=PALETTE[:len(pc2)],
        autopct='%1.0f%%', textprops={'color':'#e0e0e0','fontsize':9})
ax4.set_title('PTP Status Mix', color='#e0e0e0')

ax5 = fig.add_subplot(gs[1, 2]); ax5.set_facecolor('#1a1d27')
sc = comp['severity'].value_counts()
ax5.barh(sc.index, sc.values, color=['#f44336','#ff7043','#ffb74d','#81c784'])
ax5.set_title('Complaints by Severity', color='#e0e0e0')
ax5.tick_params(colors='#b0b0b0'); ax5.grid(axis='x', alpha=0.3)

plt.savefig('credresolve_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print('Dashboard saved.')


---
## Appendix: Data Quality Caveats

| Issue | Impact | Mitigation Applied |
|---|---|---|
| Event data 7.5 months, not 12 | MoM trend incomplete | Disclosed; Aug partial month excluded |
| 3,745 repeated payment_references | Over-count risk | Multi-criteria dedup plus reversal flags |
| Mixed timezones UTC/IST/GST | Cross-month assignment errors | Disclosed; normalization recommended |
| 600 exact duplicate borrower rows | Inflated borrower counts | De-duplicated before entity analysis |
| Legacy plus current disposition codes | PTP identification gap | Both PTP and PROMISE_TO_PAY codes used |
| Suspended agents in call data | Compliance flag | Identified; audit recommended |

## Caveats on the 11% MoM Claim

1. No fixed denominator: changing eligible pool makes rates incomparable month-to-month
2. Mix effects: growth in easier-to-collect segments could inflate headline metric
3. Timezone ambiguity: naive timestamps may assign payments to wrong months
4. Reversal lag: reversals in a later month can inflate an earlier month net figure

The 11% claim requires all above issues resolved before confident assertion.
